In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from src import config
from src.data_loader import load_images_from_folder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1) Carrega treino e teste
X, y, class_names = load_images_from_folder(config.TRAIN_DIR)
print("Treino (bruto):", X.shape, "| Classes:", class_names)

X_test, y_test, _ = load_images_from_folder(config.TEST_DIR)
print("Teste (bruto):", X_test.shape)

# 2) Separa treino/validacao (80/20, estratificado)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=config.RANDOM_STATE,
)
print("Treino:", X_train.shape, "| Validacao:", X_val.shape)

# 3) Normaliza -- fit SOMENTE no treino
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 4) PCA -- fit SOMENTE no treino
pca = PCA(n_components=150, random_state=config.RANDOM_STATE)
X_train = pca.fit_transform(X_train)
X_val = pca.transform(X_val)
X_test = pca.transform(X_test)
print(f"Variancia explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print("Shape final -> treino:", X_train.shape, "| validacao:", X_val.shape, "| teste:", X_test.shape)

# 5) Funcao de avaliacao -- reusada em todos os experimentos
def evaluate_model(model, X, y_true, rotulo):
    y_pred = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    print(f"  [{rotulo:6s}] Acuracia: {metrics['accuracy']:.4f} | Precisao: {metrics['precision']:.4f} | "
          f"Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f}")
    return metrics

# Resultados do grupo MLP (Exp1-4), ja confirmados por voce -- fixos aqui so pra
# a tabela final ficar completa, sem precisar re-treinar o MLP de novo
results = [
    {"Experimento": "Exp1 - MLP Baseline", "accuracy": 0.840625, "precision": 0.841341, "recall": 0.840625, "f1_score": 0.839096},
    {"Experimento": "Exp2 - MLP + L2 forte", "accuracy": 0.860000, "precision": 0.860370, "recall": 0.860000, "f1_score": 0.857450},
    {"Experimento": "Exp3 - MLP + Early Stopping", "accuracy": 0.821250, "precision": 0.822044, "recall": 0.821250, "f1_score": 0.819237},
    {"Experimento": "Exp4 - MLP Melhor Esforco", "accuracy": 0.861250, "precision": 0.862824, "recall": 0.861250, "f1_score": 0.859499},
]

# ============================================================
# Experimento 5: SVM Baseline (kernel linear)
# ============================================================
print("\n=== Experimento 5 - SVM Baseline (kernel linear, C=1.0) ===")
svm_5 = SVC(kernel="linear", C=1.0, random_state=config.RANDOM_STATE)
svm_5.fit(X_train, y_train)
evaluate_model(svm_5, X_train, y_train, "TREINO")
metrics_5 = evaluate_model(svm_5, X_test, y_test, "TESTE")
results.append({"Experimento": "Exp5 - SVM Baseline (linear)", **metrics_5})

# ============================================================
# Tabela acumulada (Exp1-5)
# ============================================================
print("\n=== Resultados acumulados (Exp1-5) ===")
print(pd.DataFrame(results))

Carregando 'glioma' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/1400 (0.3s decorridos)
  ... 600/1400 (0.5s decorridos)
  ... 800/1400 (0.7s decorridos)
  ... 1000/1400 (0.9s decorridos)
  ... 1200/1400 (1.1s decorridos)
  ... 1400/1400 (1.2s decorridos)
  'glioma' concluida em 1.2s
Carregando 'meningioma' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/1400 (0.4s decorridos)
  ... 600/1400 (0.6s decorridos)
  ... 800/1400 (0.9s decorridos)
  ... 1000/1400 (1.1s decorridos)
  ... 1200/1400 (1.3s decorridos)
  ... 1400/1400 (1.5s decorridos)
  'meningioma' concluida em 1.5s
Carregando 'notumor' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/1400 (0.3s decorridos)
  ... 600/1400 (0.5s decorridos)
  ... 800/1400 (0.7s decorridos)
  ... 1000/1400 (0.8s decorridos)
  ... 1200/1400 (1.0s decorridos)
  ... 1400/1400 (1.2s decorridos)
  'notumor' concluida em 1.2s
Carregando 'pituitary' (1400 imagens)...
  ... 200/1400 (0.2s decorridos)
  ... 400/140